# Lab 4 — Multi-Layer Perceptron (Keras) for Multiclass Classification

Building a Multi-Layer Perceptron (MLP) using Keras (`tf.keras`) to classify Iris flowers into 3 categories (Setosa, Versicolor, Virginica).

**Network Architecture**
```
Input (4 features)
    -> Dense Layer 1 - 16 neurons + ReLU
    -> Dense Layer 2 - 8 neurons + ReLU
    -> Output - 3 classes (Softmax)
```

**Key Concepts**
| Concept | What it means (simply) |
|---|---|
| MLP | A feedforward network with one or more hidden Dense layers |
| Softmax | Output activation that turns logits into class probabilities that sum to 1 |
| One-hot encoding | Converts integer class labels into binary vectors for `categorical_crossentropy` |
| Categorical Crossentropy | Loss function for multi-class classification with one-hot labels |
| Adam Optimizer | Adaptive gradient-based optimizer used to train the network |
| Epoch / Batch | One full pass over the data / one chunk of samples processed per weight update |

## 1. Load and Prepare Data

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

tf.random.set_seed(42)
np.random.seed(42)

iris = load_iris()
X = iris.data        # Features: sepal length, sepal width, petal length, petal width
y = iris.target       # Labels: 0 = Setosa, 1 = Versicolor, 2 = Virginica
class_names = iris.target_names

# Normalize features (zero mean, unit variance)
scaler = StandardScaler()
X = scaler.fit_transform(X)

# One-hot encode the labels (needed for categorical_crossentropy)
encoder = OneHotEncoder(sparse_output=False)
y_onehot = encoder.fit_transform(y.reshape(-1, 1))

# Split into train (80%) and test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_onehot, test_size=0.2, random_state=42, stratify=y
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

Train shape: (120, 4), Test shape: (30, 4)


## 2. Build the Model

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(4,)),
    layers.Dense(16, activation="relu"),   # Hidden layer 1
    layers.Dense(8, activation="relu"),    # Hidden layer 2
    layers.Dense(3, activation="softmax")  # Output layer - probabilities over 3 classes
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            27 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 243 (972.00 B)

 Trainable params: 243 (972.00 B)

 Non-trainable params: 0 (0.00 B)

## 3. Compile

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

## 4. Train the Model

In [8]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=8,
    verbose=0
)

for epoch in range(9, 100, 10):
    print(f"Epoch [{epoch+1:3d}/100] | Train Loss: {history.history['loss'][epoch]:.4f} | "
          f"Test Loss: {history.history['val_loss'][epoch]:.4f} | "
          f"Test Accuracy: {history.history['val_accuracy'][epoch]*100:.1f}%")

Epoch [ 10/100] | Train Loss: 0.0009 | Test Loss: 0.2163 | Test Accuracy: 93.3%
Epoch [ 20/100] | Train Loss: 0.0007 | Test Loss: 0.2293 | Test Accuracy: 93.3%
Epoch [ 30/100] | Train Loss: 0.0006 | Test Loss: 0.2330 | Test Accuracy: 93.3%
Epoch [ 40/100] | Train Loss: 0.0005 | Test Loss: 0.2424 | Test Accuracy: 93.3%
Epoch [ 50/100] | Train Loss: 0.0004 | Test Loss: 0.2459 | Test Accuracy: 93.3%
Epoch [ 60/100] | Train Loss: 0.0003 | Test Loss: 0.2544 | Test Accuracy: 93.3%
Epoch [ 70/100] | Train Loss: 0.0003 | Test Loss: 0.2611 | Test Accuracy: 93.3%
Epoch [ 80/100] | Train Loss: 0.0002 | Test Loss: 0.2669 | Test Accuracy: 93.3%
Epoch [ 90/100] | Train Loss: 0.0002 | Test Loss: 0.2743 | Test Accuracy: 93.3%
Epoch [100/100] | Train Loss: 0.0002 | Test Loss: 0.2754 | Test Accuracy: 93.3%


## 5. Final Evaluation

In [9]:
print("\n-- Final Results --")
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
correct = int(round(test_accuracy * len(X_test)))
print(f"Test Accuracy: {test_accuracy*100:.1f}%  ({correct}/{len(X_test)} correct)")

print("\nSample Predictions vs Actual:")
predictions = model.predict(X_test, verbose=0)
for i in range(5):
    predicted = class_names[np.argmax(predictions[i])]
    actual = class_names[np.argmax(y_test[i])]
    print(f"  Predicted: {predicted:12s} | Actual: {actual}")


-- Final Results --
Test Accuracy: 93.3%  (28/30 correct)

Sample Predictions vs Actual:
  Predicted: setosa       | Actual: setosa
  Predicted: virginica    | Actual: virginica
  Predicted: versicolor   | Actual: versicolor
  Predicted: versicolor   | Actual: versicolor
  Predicted: setosa       | Actual: setosa
